# SBA 7(a) Lender Underwriting Analysis

Which lenders underwrite better than their loan book would predict?

Data: SBA 7(a) FOIA loan-level file, FY2000-FY2009 (https://data.sba.gov/dataset/7-a-504-foia)

Raw default rates mostly measure *what and when* a lender lent, not how well it underwrote.
This notebook builds an expected default rate for each lender from the year-and-sector
composition of its own portfolio, compares it to actual performance, and then tests the
result with a logistic regression controlling for loan characteristics.

## Setup

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size'] = 10

RED = '#C0392B'
GREEN = '#1D8348'
GREY = '#767676'

conn = sqlite3.connect('sba.db')

## 1. The working population

`LoanStatus` distinguishes resolved from unresolved loans:

- `P I F` — paid in full (note the spaces in the stored value)
- `CHGOFF` — charged off
- `CANCLD` — approved but never funded
- `COMMIT` — undisbursed
- `EXEMPT` — outstanding, withheld under FOIA Exemption 4

Only `CHGOFF` and `P I F` are outcomes. Everything below filters to those two.

In [ ]:
status = pd.read_sql_query(
    "SELECT LoanStatus, COUNT(*) AS loans FROM loans GROUP BY LoanStatus ORDER BY loans DESC",
    conn
)
status

Only ~3,500 loans are `EXEMPT` out of ~690,000, so censoring is negligible for this decade —
loans from the 2000s have had 15+ years to resolve. That is the main reason for choosing this
file over a more recent one.

## 2. The vintage effect

In [ ]:
vintage = pd.read_sql_query('''
SELECT
    ApprovalFY,
    COUNT(*) AS resolved_loans,
    SUM(CASE WHEN LoanStatus = 'CHGOFF' THEN 1.0 ELSE 0 END) / COUNT(*) * 100 AS default_rate_pct
FROM loans
WHERE LoanStatus IN ('CHGOFF', 'P I F')
GROUP BY ApprovalFY
ORDER BY ApprovalFY
''', conn)

vintage

In [ ]:
fig, ax = plt.subplots()
ax.plot(vintage['ApprovalFY'], vintage['default_rate_pct'],
        marker='o', color=RED, linewidth=2)
ax.set_xlabel('Fiscal year loan was approved')
ax.set_ylabel('Default rate (%)')
ax.set_title('Default rate by origination year', loc='left', fontsize=12)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 40)

peak = vintage.loc[vintage['default_rate_pct'].idxmax()]
ax.annotate(f"{peak['default_rate_pct']:.1f}%",
            xy=(peak['ApprovalFY'], peak['default_rate_pct']),
            xytext=(0, 8), textcoords='offset points',
            ha='center', color=RED)
plt.tight_layout()
plt.show()

Default rates nearly tripled from 13.5% (FY2002) to 37.0% (FY2007), then fell back to
14.9% by FY2009. The peak is in loans *originated* in 2006–2007, which hit the recession
partway through their term — not in loans originated during the recession itself, when
lending standards had already tightened and volume had halved.

**Vintage is the single largest driver of outcome, so any lender comparison has to control for it.**

## 3. Industry, and whether it is really industry

In [ ]:
sector = pd.read_sql_query('''
SELECT
    SUBSTR(NaicsCode, 1, 2) AS sector,
    COUNT(*) AS loans,
    SUM(CASE WHEN LoanStatus = 'CHGOFF' THEN 1.0 ELSE 0 END) / COUNT(*) * 100 AS default_rate_pct
FROM loans
WHERE LoanStatus IN ('CHGOFF', 'P I F')
GROUP BY SUBSTR(NaicsCode, 1, 2)
HAVING COUNT(*) >= 1000
ORDER BY default_rate_pct DESC
''', conn)

NAICS = {
    '11':'Agriculture','21':'Mining','22':'Utilities','23':'Construction',
    '31':'Manufacturing','32':'Manufacturing','33':'Manufacturing',
    '42':'Wholesale trade','44':'Retail trade','45':'Retail trade',
    '48':'Transportation','49':'Warehousing','51':'Information',
    '52':'Finance & insurance','53':'Real estate','54':'Professional services',
    '55':'Management','56':'Admin & waste services','61':'Education',
    '62':'Health care','71':'Arts & recreation','72':'Accommodation & food',
    '81':'Other services','92':'Public administration',
}
sector['name'] = sector['sector'].map(NAICS).fillna('(blank NAICS)')
sector

In [ ]:
s = sector.dropna(subset=['sector']).sort_values('default_rate_pct')
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(s['name'], s['default_rate_pct'], color=GREY)
ax.axvline(23.9, color=RED, linestyle='--', linewidth=1)
ax.text(24.4, 0.2, 'overall 23.9%', color=RED, fontsize=9)
ax.set_xlabel('Default rate (%)')
ax.set_title('Default rate by industry sector', loc='left', fontsize=12)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

Real estate and construction lead; health care and mining sit at the bottom.

But real estate and construction lending was *concentrated in exactly the years that failed*,
so the ranking above may be a vintage artifact wearing an industry costume. Breaking it out
by year tests that.

In [ ]:
interaction = pd.read_sql_query('''
SELECT
    ApprovalFY,
    SUBSTR(NaicsCode, 1, 2) AS sector,
    COUNT(*) AS loans,
    SUM(CASE WHEN LoanStatus = 'CHGOFF' THEN 1.0 ELSE 0 END) / COUNT(*) * 100 AS default_rate_pct
FROM loans
WHERE LoanStatus IN ('CHGOFF', 'P I F')
  AND SUBSTR(NaicsCode, 1, 2) IN ('53', '23', '62')
GROUP BY SUBSTR(NaicsCode, 1, 2), ApprovalFY
HAVING COUNT(*) >= 200
ORDER BY sector, ApprovalFY
''', conn)

pivot = interaction.pivot(index='ApprovalFY', columns='sector', values='default_rate_pct')
pivot.columns = [NAICS[c] for c in pivot.columns]
pivot.round(1)

In [ ]:
fig, ax = plt.subplots()
colors = {'Real estate': RED, 'Construction': '#E67E22', 'Health care': GREEN}
for col in pivot.columns:
    ax.plot(pivot.index, pivot[col], marker='o', label=col,
            color=colors.get(col, GREY), linewidth=2)
ax.set_xlabel('Fiscal year loan was approved')
ax.set_ylabel('Default rate (%)')
ax.set_title('Every sector tripled — the base rate is what differs',
             loc='left', fontsize=12)
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

All three roughly tripled from trough to peak. The *multiplier* is similar across sectors;
the *base rate* is what separates them. Health care's worst year (23.1%) is close to real
estate's average year — it is not cycle-proof, it just starts from a much safer floor.

That is a more useful finding than the flat ranking, and it means both year and sector belong
in the benchmark.

## 4. Lender comparison — first attempt

For each lender: compute the default rate expected from the year-and-sector mix of its own
portfolio, and compare to what it actually experienced.

In [ ]:
BENCHMARK_CTE = '''
WITH benchmark AS (
    SELECT
        ApprovalFY,
        SUBSTR(NaicsCode, 1, 2) AS sector,
        SUM(CASE WHEN LoanStatus = 'CHGOFF' THEN 1.0 ELSE 0 END) / COUNT(*) * 100 AS expected_rate
    FROM loans
    WHERE LoanStatus IN ('CHGOFF', 'P I F')
    GROUP BY ApprovalFY, SUBSTR(NaicsCode, 1, 2)
    HAVING COUNT(*) >= 200
)
'''

lenders_v1 = pd.read_sql_query(BENCHMARK_CTE + '''
SELECT
    l.BankName,
    COUNT(*) AS loans,
    SUM(CASE WHEN l.LoanStatus = 'CHGOFF' THEN 1.0 ELSE 0 END) / COUNT(*) * 100 AS actual_pct,
    AVG(b.expected_rate) AS expected_pct,
    SUM(CASE WHEN l.LoanStatus = 'CHGOFF' THEN 1.0 ELSE 0 END) / COUNT(*) * 100
        - AVG(b.expected_rate) AS difference
FROM loans l
JOIN benchmark b
    ON l.ApprovalFY = b.ApprovalFY
   AND SUBSTR(l.NaicsCode, 1, 2) = b.sector
WHERE l.LoanStatus IN ('CHGOFF', 'P I F')
GROUP BY l.BankName
HAVING COUNT(*) >= 500
ORDER BY difference
''', conn)

lenders_v1.head(8).round(2)

**These results cannot be true.**

Citibank (West), FSB shows a 0.00% default rate across 574 loans. Wachovia SBA Lending shows
0.16% across 1,286. The base rate for this decade is 23.9%, and it includes the financial crisis.

The data dictionary explains why: `BankName` is *"the bank that the loan is currently assigned
to"* — not the lender that originated it. Loans move between institutions through mergers,
acquisitions, and failures.

## 5. Diagnosing the reassignment problem

In [ ]:
diagnostic = pd.read_sql_query('''
SELECT
    BankName,
    ApprovalFY,
    COUNT(*) AS loans,
    SUM(CASE WHEN LoanStatus = 'CHGOFF' THEN 1 ELSE 0 END) AS charged_off
FROM loans
WHERE BankName IN (
        'Citibank (West), FSB',
        'Wachovia SBA Lending, Inc.',
        'Federal Deposit Insurance Corporation'
      )
  AND LoanStatus IN ('CHGOFF', 'P I F')
GROUP BY BankName, ApprovalFY
ORDER BY BankName, ApprovalFY
''', conn)

diagnostic.pivot(index='ApprovalFY', columns='BankName',
                 values=['loans', 'charged_off']).fillna(0).astype(int)

Three things confirm the mechanism:

1. **Citibank (West) and Wachovia both stop originating after FY2007.** Citibank (West) was
   consolidated into Citibank N.A.; Wachovia was absorbed by Wells Fargo in 2008.
2. **Neither records a single charge-off in any crisis year.** Roughly 2,000 loans between them,
   two failures.
3. **The FDIC appears as a "lender" with 4,058 loans** spanning all ten years, with charge-offs
   rising through the crisis as expected — despite originating nothing. It only holds loans
   inherited from banks that failed.

When these institutions dissolved, charged-off loans were reassigned elsewhere while performing
loans kept the legacy name. What survives under the defunct entity is a survivorship-filtered
remnant.

**The bias is asymmetric.** Apparent *top* performers are the suspect ones — nobody acquires a
book of charge-offs by choice. High default rates are not produced by this artifact.

## 6. Lender comparison — corrected

Two changes:

- require at least 50 originations in **both** FY2003 and FY2008, so only lenders that operated
  across the full period appear
- exclude the FDIC by name

In [ ]:
lenders = pd.read_sql_query(BENCHMARK_CTE + '''
SELECT
    l.BankName,
    COUNT(*) AS loans,
    SUM(CASE WHEN l.LoanStatus = 'CHGOFF' THEN 1.0 ELSE 0 END) / COUNT(*) * 100 AS actual_pct,
    AVG(b.expected_rate) AS expected_pct,
    SUM(CASE WHEN l.LoanStatus = 'CHGOFF' THEN 1.0 ELSE 0 END) / COUNT(*) * 100
        - AVG(b.expected_rate) AS difference
FROM loans l
JOIN benchmark b
    ON l.ApprovalFY = b.ApprovalFY
   AND SUBSTR(l.NaicsCode, 1, 2) = b.sector
WHERE l.LoanStatus IN ('CHGOFF', 'P I F')
  AND l.BankName != 'Federal Deposit Insurance Corporation'
GROUP BY l.BankName
HAVING COUNT(*) >= 500
   AND SUM(CASE WHEN l.ApprovalFY = 2003 THEN 1 ELSE 0 END) >= 50
   AND SUM(CASE WHEN l.ApprovalFY = 2008 THEN 1 ELSE 0 END) >= 50
ORDER BY difference
''', conn)

print(f"{len(lenders)} lenders")
print(f"spread: {lenders['difference'].max() - lenders['difference'].min():.1f} percentage points")
lenders.round(2).head(10)

In [ ]:
top = lenders.nsmallest(15, 'difference')
bottom = lenders.nlargest(15, 'difference')
combined = pd.concat([bottom.iloc[::-1], top])

fig, ax = plt.subplots(figsize=(9, 10))
colors = [RED if d > 0 else GREEN for d in combined['difference']]
labels = [f"{n[:34]}  ({l:,})" for n, l in zip(combined['BankName'], combined['loans'])]
ax.barh(labels, combined['difference'], color=colors)
ax.axvline(0, color='#333333', linewidth=0.8)
ax.set_xlabel('Actual minus expected default rate (percentage points)')
ax.set_title('Lender performance vs. its own portfolio mix\nbest 15 and worst 15, loan count in parentheses',
             loc='left', fontsize=12)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

A **34-point spread** separates the best and worst underwriters after controlling for
industry and vintage.

Mid-size regional banks occupy most of the top of the list. Scale does not explain the outcome
on its own — U.S. Bank (25,144 loans) and KeyBank (7,819) are large and outperformed;
Bank of America (68,606) and Capital One (19,477) are large and did not.

## 7. Does geography explain it?

Before adding state as a third benchmark dimension — which would mean ~11,000 mostly-sparse
year × sector × state cells — it is worth checking whether geography is doing any work at all.

In [ ]:
def by_state(bank, n=12):
    return pd.read_sql_query(f'''
    SELECT BorrState,
           COUNT(*) AS loans,
           SUM(CASE WHEN LoanStatus = 'CHGOFF' THEN 1.0 ELSE 0 END) / COUNT(*) * 100 AS default_rate_pct
    FROM loans
    WHERE BankName = "{bank}"
      AND LoanStatus IN ('CHGOFF', 'P I F')
    GROUP BY BorrState
    HAVING COUNT(*) >= 50
    ORDER BY loans DESC
    LIMIT {n}
    ''', conn)

worst_bank = by_state('Bank of Hope')
best_bank = by_state('Commerce Bank')

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True)
for ax, data, name, color in [
    (axes[0], worst_bank, 'Bank of Hope (worst overall)', RED),
    (axes[1], best_bank, 'Commerce Bank (best overall)', GREEN),
]:
    d = data.sort_values('default_rate_pct')
    ax.barh(d['BorrState'], d['default_rate_pct'], color=color)
    ax.axvline(23.9, color='#333333', linestyle='--', linewidth=1)
    ax.set_title(name, loc='left', fontsize=11)
    ax.set_xlabel('Default rate (%)')
    ax.grid(axis='x', alpha=0.3)
axes[0].text(24.5, 0.2, 'national 23.9%', fontsize=8, color='#333333')
plt.tight_layout()
plt.show()

**Geography does not explain the spread.**

Bank of Hope defaults above the national base rate in essentially every state it lends in,
including Texas and Georgia — not just the collapsed housing markets of California, Nevada,
and Florida. Poor outcomes follow the lender across 15+ states.

Commerce Bank runs 12–13% across ordinary Midwest markets.

The benchmark was therefore left at two dimensions. **Recording a control that was tested and
rejected is part of the method, not an omission.**

## 8. Regression test

The SQL shows large differences. A logistic regression tests whether lender identity still
carries information once loan-level characteristics are controlled for *simultaneously*
rather than sequentially.

In [ ]:
df = pd.read_sql_query('''
SELECT
    BankName,
    ApprovalFY,
    SUBSTR(NaicsCode, 1, 2) AS sector,
    GrossApproval,
    SBAGuaranteedApproval,
    TermInMonths,
    BusinessAge,
    CollateralInd,
    RevolverStatus,
    CASE WHEN LoanStatus = 'CHGOFF' THEN 1 ELSE 0 END AS defaulted
FROM loans
WHERE LoanStatus IN ('CHGOFF', 'P I F')
''', conn)

print(df.shape)
print(f"base default rate: {df['defaulted'].mean():.3f}")
df.isnull().sum()

### Two variables that run backwards

In [ ]:
for col in ['CollateralInd', 'RevolverStatus']:
    print(df.groupby(col)['defaulted'].agg(['mean', 'count']).round(3), '\n')

**Collateralized loans default at 32.3% versus 22.4% uncollateralized** — ten points *worse*.

Collateral is supposed to reduce risk, so this looks wrong. It is reverse causality: lenders
require collateral when they judge a borrower risky. The variable marks the lender's own risk
assessment rather than a protective feature.

This is worth flagging before modelling, because the coefficient will carry the same sign and
should not be read as "collateral causes default."

In [ ]:
model_df = df[df['sector'].notna()].copy()

model_df['collateral'] = (model_df['CollateralInd'] == 'Y').astype(int)
model_df['revolver'] = (model_df['RevolverStatus'] == 'Y').astype(int)
model_df['guarantee_pct'] = model_df['SBAGuaranteedApproval'] / model_df['GrossApproval']
model_df['log_amount'] = np.log(model_df['GrossApproval'])

def age_group(x):
    if not isinstance(x, str):
        return 'other'
    if 'New' in x or 'Startup' in x:
        return 'new'
    if '5 or more' in x:
        return 'established'
    return 'other'

model_df['business_age_grp'] = model_df['BusinessAge'].map(age_group)

print(model_df['business_age_grp'].value_counts())
model_df['guarantee_pct'].describe().round(3)

Note `guarantee_pct` clusters at 0.50 and 0.75 rather than varying smoothly — these are
program tiers (SBA Express carries 50%, standard 7(a) 75–85%), not a continuous risk measure.

Note also that nearly half of all loans fall into a single `BusinessAge` category
("less than 4 years old but at least 3"), which is implausible as a real distribution and
suggests a default value rather than a measurement. It is kept in the model but should not be
trusted.

In [ ]:
# restrict to the same 73 lenders as the SQL analysis
counts = model_df['BankName'].value_counts()
big = set(counts[counts >= 500].index)

early = model_df[model_df['ApprovalFY'] == 2003].groupby('BankName').size()
late = model_df[model_df['ApprovalFY'] == 2008].groupby('BankName').size()
survivors = set(early[early >= 50].index) & set(late[late >= 50].index)

sub = model_df[
    model_df['BankName'].isin(big & survivors)
    & (model_df['BankName'] != 'Federal Deposit Insurance Corporation')
].copy()

print(f"{len(sub):,} loans | {sub['BankName'].nunique()} lenders")

In [ ]:
NUMERIC = ['log_amount', 'TermInMonths', 'guarantee_pct',
           'collateral', 'revolver', 'ApprovalFY']

def build_X(d, with_lender):
    cat = ['sector', 'business_age_grp'] + (['BankName'] if with_lender else [])
    X = pd.get_dummies(d[cat], drop_first=True).astype(float)
    for c in NUMERIC:
        X[c] = d[c].values
    return sm.add_constant(X)

y = sub['defaulted']
m_without = sm.Logit(y, build_X(sub, False)).fit(disp=0)
m_with = sm.Logit(y, build_X(sub, True)).fit(disp=0)

results = pd.DataFrame({
    'model': ['Loan characteristics only', 'Loan characteristics + lender'],
    'pseudo_r2': [m_without.prsquared, m_with.prsquared],
})
results['improvement'] = results['pseudo_r2'].diff()
results.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Loan\ncharacteristics', 'Loan characteristics\n+ lender identity'],
              results['pseudo_r2'], color=[GREY, GREEN], width=0.5)
for b, v in zip(bars, results['pseudo_r2']):
    ax.text(b.get_x() + b.get_width()/2, v + 0.004, f'{v:.3f}',
            ha='center', fontsize=10)
ax.set_ylabel('Pseudo R-squared')
ax.set_ylim(0, 0.30)
ax.set_title('Lender identity adds real explanatory power', loc='left', fontsize=12)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

pct = (results['pseudo_r2'].iloc[1] / results['pseudo_r2'].iloc[0] - 1) * 100
print(f"Adding lender identity improves explanatory power by {pct:.0f}%")

Knowing **which institution** originated a loan carries substantial information beyond
sector, business age, loan size, term, guarantee percentage, collateral, revolver status, and
vintage combined.

That is what the SQL comparison implied, tested a different way.

### Coefficients on the loan characteristics

In [ ]:
coefs = m_with.params[~m_with.params.index.str.startswith(('BankName', 'const'))]
coefs = coefs.sort_values()

fig, ax = plt.subplots(figsize=(9, 9))
labels = [NAICS.get(c.replace('sector_', ''), c) if c.startswith('sector_') else c
          for c in coefs.index]
ax.barh(labels, coefs.values,
        color=[RED if v > 0 else GREEN for v in coefs.values])
ax.axvline(0, color='#333333', linewidth=0.8)
ax.set_xlabel('Log-odds coefficient (positive = higher default risk)')
ax.set_title('What predicts default, controlling for lender', loc='left', fontsize=12)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

coefs.round(3)

**Guarantee percentage is the strongest single predictor (+2.37).** This is not the SBA
guarantee causing failure. Guarantee percentage determines how much of the loss the lender
absorbs — a lender retaining 25% of the risk has weaker incentive to screen than one retaining
50%. Moral hazard is a plausible reading; causation is not established.

**Collateral predicts default (+0.29), not safety** — the reverse-causality pattern identified
before modelling, surviving all controls.

**Longer terms are protective (−0.035).** Partly loan type: long-term loans are typically
real-estate-secured with lower payments, while short-term working capital lending is riskier.

**Business age contributes almost nothing** (coefficients near zero). Given how implausible that
field's distribution looked, this more likely reflects an unreliable variable than a genuine
absence of effect.

**Sector coefficients reproduce the SQL ranking** — mining and health care most protective,
real estate and accommodation/food service worst. Two independent methods agreeing is
reassuring.

## What this analysis does not establish

- **Cause.** A lender above or below its benchmark could differ on underwriting standards,
  borrower selection, servicing and workout practices, regional exposure not captured by state,
  or business lines this data does not distinguish.
- **Originator attribution.** Even after filtering, `BankName` reflects current assignment.
  Lenders surviving the period intact are less affected, but some portfolio movement remains.
- **Unobserved loan quality.** Loan purpose, borrower credit score, and collateral *value* are
  not in this data.
- **One decade, one program.** FY2000–2009 7(a) only, spanning an unusually severe credit cycle.
  Whether these differences persist in calmer periods is untested.
- **Missing industry data.** 16,884 loans (2.8%) have a blank NAICS code and are excluded from
  anything sector-based.

In [ ]:
conn.close()